# World Cup Predictor — Analysis Notebook

Run `wc_create_tables.py` then `wc_etl.py` before executing cells here.

In [ ]:
import psycopg2
import pandas as pd

conn = psycopg2.connect('host=127.0.0.1 dbname=worldcupdb user=student password=student')
cur  = conn.cursor()
print('Connected.')

## Group Standings

In [ ]:
from wc_sql_queries import group_standings_select

cur.execute(group_standings_select)
cols = [d[0] for d in cur.description]
df_standings = pd.DataFrame(cur.fetchall(), columns=cols)
df_standings

## Group Stage Results

In [ ]:
from wc_sql_queries import group_matches_select

cur.execute(group_matches_select)
cols = [d[0] for d in cur.description]
df_matches = pd.DataFrame(cur.fetchall(), columns=cols)
df_matches

## Top Goal Scorers

In [ ]:
from wc_sql_queries import top_scorers_select

cur.execute(top_scorers_select)
cols = [d[0] for d in cur.description]
df_scorers = pd.DataFrame(cur.fetchall(), columns=cols)
df_scorers

## Match Predictions

In [ ]:
from wc_sql_queries import predictions_select

cur.execute(predictions_select)
cols = [d[0] for d in cur.description]
df_preds = pd.DataFrame(cur.fetchall(), columns=cols)
df_preds.head(20)

## Prediction Accuracy (Group Stage)

In [ ]:
accuracy_sql = """
SELECT
    COUNT(*) AS total_matches,
    SUM(CASE
        WHEN pr.predicted_winner_id = m.winner_id THEN 1
        WHEN pr.predicted_winner_id IS NULL AND m.winner_id IS NULL THEN 1
        ELSE 0
    END) AS correct_predictions,
    ROUND(
        100.0 * SUM(CASE
            WHEN pr.predicted_winner_id = m.winner_id THEN 1
            WHEN pr.predicted_winner_id IS NULL AND m.winner_id IS NULL THEN 1
            ELSE 0
        END) / COUNT(*), 2
    ) AS accuracy_pct
FROM predictions pr
JOIN matches m ON pr.match_id = m.match_id
WHERE m.stage = 'Group Stage';
"""

cur.execute(accuracy_sql)
cols = [d[0] for d in cur.description]
df_acc = pd.DataFrame(cur.fetchall(), columns=cols)
df_acc

## Teams by FIFA Ranking

In [ ]:
cur.execute('SELECT name, fifa_code, group_name, ranking FROM teams ORDER BY ranking')
cols = [d[0] for d in cur.description]
df_teams = pd.DataFrame(cur.fetchall(), columns=cols)
df_teams

In [ ]:
conn.close()